# 🔮 Stage 4 — Missing Value Imputation

**Goal:** Fill missing fields using principled imputation strategies.

| Field | Strategy | Rationale |
|-------|----------|-----------|
| `prep_time_minutes` | **sklearn Median** | Robust to long-cook outliers (brisket ≠ smoothie) |
| `servings` | **Rule-based default (4)** | Standard household recipe assumption |

This notebook covers:
- Visualising missing values before imputation
- Running sklearn `SimpleImputer(strategy='median')`
- Before / after comparison
- Imputation audit log
- Saving `imputed_data.json` and `imputation_log.json`

---


## 0. Setup

In [ ]:
import sys, json, os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.impute import SimpleImputer
from src.imputation import impute_missing_values, build_imputation_log
from src.config import DEFAULT_SERVINGS

INPUT_FILE   = os.path.join(REPO_ROOT, 'data', 'enriched_data.json')
OUTPUT_DATA  = os.path.join(REPO_ROOT, 'data', 'imputed_data.json')
OUTPUT_LOG   = os.path.join(REPO_ROOT, 'data', 'imputation_log.json')

print('Setup complete.')


In [ ]:
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    recipes = json.load(f)
print(f'Loaded {len(recipes)} enriched recipes.')


## 1. Pre-Imputation State

In [ ]:
pre_df = pd.DataFrame([{
    'recipe_id':         r['recipe_id'],
    'name':              r['name'],
    'prep_time_minutes': r.get('prep_time_minutes'),
    'servings':          r.get('servings'),
} for r in recipes])

print('Missing counts before imputation:')
print(pre_df[['prep_time_minutes','servings']].isnull().sum())
print()
pre_df


## 2. Prep-Time Distribution (known values only)

In [ ]:
known_times = pre_df['prep_time_minutes'].dropna().values
median_time = float(np.median(known_times)) if len(known_times) else 30.0

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(known_times, bins=8, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(median_time, color='red', linestyle='--', linewidth=2, label=f'Median = {median_time:.0f} min')
ax.axvline(float(np.mean(known_times)), color='orange', linestyle='--', linewidth=2,
           label=f'Mean = {np.mean(known_times):.1f} min')
ax.set_xlabel('Prep Time (minutes)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Known Prep Times\n(red dashed = imputation target)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'data', 'prep_time_dist_pre.png'), dpi=120)
plt.show()
print(f'Median: {median_time:.0f} min  |  Mean: {np.mean(known_times):.1f} min  |  Std: {np.std(known_times):.1f} min')


> **Why median and not mean?**  
> The Beef Stew (120 min) pulls the mean up significantly. Median is insensitive to such outliers,
> making it a better central estimate for the typical missing recipe.

## 3. sklearn SimpleImputer Demo

In [ ]:
times_arr = np.array([r.get('prep_time_minutes') for r in recipes], dtype=float)
imputer   = SimpleImputer(strategy='median')
imputed   = imputer.fit_transform(times_arr.reshape(-1, 1)).flatten()

print(f'sklearn computed median = {imputer.statistics_[0]:.1f} min\n')
print('Before vs After imputation (prep_time_minutes):')
for i, r in enumerate(recipes):
    before = r.get('prep_time_minutes')
    after  = int(round(imputed[i]))
    flag   = ' ← IMPUTED' if before is None else ''
    print(f"  {r['recipe_id']:8} before={str(before):<6} after={after}{flag}")


## 4. Run Imputation on All Recipes

In [ ]:
import copy
recipes = copy.deepcopy(recipes)  # preserve pre-state for comparison
recipes = impute_missing_values(recipes)

imputed_log = build_imputation_log(recipes)
print(f'{len(imputed_log)} recipes had values imputed.')


## 5. Before / After Comparison Table

In [ ]:
post_df = pd.DataFrame([{
    'recipe_id':         r['recipe_id'],
    'name':              r['name'],
    'prep_time_before':  r['_meta']['original_values'].get('prep_time_minutes', r['prep_time_minutes']),
    'prep_time_after':   r['prep_time_minutes'],
    'servings_before':   r['_meta']['original_values'].get('servings', r['servings']),
    'servings_after':    r['servings'],
    'imputed_fields':    ', '.join(r['_meta']['imputed_fields']) or '—',
} for r in recipes])
post_df


## 6. Imputation Audit Log

In [ ]:
pd.DataFrame(imputed_log)


## 7. Before / After Visualisation

In [ ]:
names   = [r['name'][:20] for r in recipes]
before  = [r['_meta']['original_values'].get('prep_time_minutes', r['prep_time_minutes']) for r in recipes]
after   = [r['prep_time_minutes'] for r in recipes]
was_imp = ['prep_time_minutes' in r['_meta']['imputed_fields'] for r in recipes]

x      = np.arange(len(names))
width  = 0.35
colors = ['#e74c3c' if imp else '#27ae60' for imp in was_imp]

fig, ax = plt.subplots(figsize=(12, 5))
bars_b = ax.bar(x - width/2, [t if t else 0 for t in before], width,
                label='Before (known)', color='#3498db', alpha=0.8)
bars_a = ax.bar(x + width/2, after, width,
                label='After imputation', color=colors, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Prep Time (minutes)')
ax.set_title('Prep Time: Before vs After Imputation\n(red bars = imputed values)')
known_patch  = mpatches.Patch(color='#27ae60', label='Known (unchanged)')
imp_patch    = mpatches.Patch(color='#e74c3c', label='Imputed (median)')
before_patch = mpatches.Patch(color='#3498db', label='Before (missing=0)')
ax.legend(handles=[before_patch, known_patch, imp_patch], fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'data', 'imputation_comparison.png'), dpi=120)
plt.show()


## 8. Save Imputed Data and Log

In [ ]:
with open(OUTPUT_DATA, 'w', encoding='utf-8') as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

with open(OUTPUT_LOG, 'w', encoding='utf-8') as f:
    json.dump(imputed_log, f, indent=2, ensure_ascii=False)

print(f'✅ Saved {len(recipes)} imputed recipes → {OUTPUT_DATA}')
print(f'✅ Saved imputation log ({len(imputed_log)} entries) → {OUTPUT_LOG}')
print('   Next step: run 05_analytics.ipynb')
